In [7]:
from dotenv import load_dotenv
load_dotenv()

from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_openai import OpenAIEmbeddings, ChatOpenAI
from langchain_chroma import Chroma
from langchain_core.prompts import PromptTemplate
from langchain_community.vectorstores import InMemoryVectorStore
from langchain.tools import tool
from langchain.agents import create_agent

loader=PyPDFLoader("../data/PdfData.pdf")
pdfData=loader.load()

splitter = RecursiveCharacterTextSplitter(chunk_size=1000,chunk_overlap=200)
splittedData=splitter.split_documents(pdfData)

embeddings = OpenAIEmbeddings(model="text-embedding-3-large")

vector_store=InMemoryVectorStore.from_documents(
    documents=splittedData,
    embedding=embeddings,
)




In [ ]:
@tool
def retriever_tool(query:str):

    """
        This tool can help you to retrieve the revelant data of the Pdf documents
    """
    print("tool called for",query)
    docs = vector_store.similarity_search(query=query,k=6)

    context=""
    for doc in docs:
        context = context + doc.page_content + "\n\n"

    return context

In [10]:
llm=ChatOpenAI(model="gpt-5")

system_prompt="""
you are a helpful assistent that answers questions using retrieved context.
ALWAYS use the 'retriever_tool' tool for questions requiring exterenal knowledge.
"""

agent=create_agent(
    model=llm,
    tools=[retriever_tool],
    system_prompt=system_prompt
)


query="what is Gen AI and Rag explain"
ans=agent.invoke({"messages":[{"role":"user","content":query}]})

tool called for Explain Generative AI and Retrieval-Augmented Generation (RAG) for beginners: definitions, how they work, key components, benefits, limitations, and simple examples.


In [11]:
print(ans["messages"][-1].content)

Here’s a clear, beginner-friendly overview.

What is Generative AI (GenAI)?
- AI systems that can create new content: text, images, audio, video, or code.
- Most modern GenAI models are large neural networks trained on huge datasets.
- Large Language Models (LLMs) are a major type focused on text—they predict the next tokens to produce answers, summaries, translations, code, and more.

Why LLMs sometimes fall short
- They only “know” what was in their training data and up to a certain cutoff date.
- They can be confidently wrong (hallucinations) when missing context or asked about private/internal information.

What is Retrieval-Augmented Generation (RAG)?
- An approach that combines search (retrieval) with generation.
- Instead of relying only on the model’s internal knowledge, RAG fetches relevant documents at query time and gives them to the LLM as context to ground its answer.

How a RAG system works (simplified)
- Collect and index content: PDFs, web pages, wikis, manuals, databas